In [3]:
!pip install tqdm

     ---------------------------------------- 78.4/78.4 KB 1.4 MB/s eta 0:00:00


You should consider upgrading via the 'C:\Users\User\Documents\TKU\大二AI實驗下\DL_ai\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
import torch
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch import optim
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

In [2]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root="dataset/", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="dataset/", train=False, transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=True)

cifar 10

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 定義資料轉換流程
transform = transforms.Compose([
    transforms.Resize(224),           
    transforms.ToTensor(),          
    transforms.Normalize(            
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# 下載並載入 CIFAR-10
train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)

test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False) 

$$O = \left\lfloor \frac{I - K + 2P}{S} \right\rfloor + 1$$

若為 same padding : P = (K - 1) / 2

In [ ]:
#為了方便模組化，會將模型拆成 1.features(卷積) 2.classifier(分類用的全連接層)
class AlexNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=1000):
        super(AlexNet, self).__init__()
        #1. features
        self.features = nn.Sequential(
            #conv1:
            nn.Conv2d(in_channels, 64, kernel_size=11, stride=4, padding=2), #224x224 -> 55x55
            nn.ReLU(inplace=True),#激活函數：擬合非線性複雜資料 f(x) = max(0, x)
            nn.MaxPool2d(kernel_size=3, stride=2), #55x55 -> 27x27

            #conv2(same padding):
            nn.Conv2d(64, 192, kernel_size=5, padding=2), #27x27 -> 27x27
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2), #27x27 -> 13x13
            #conv3(same padding), 4(same padding), 5:
            nn.Conv2d(192, 384, kernel_size=3, padding=1), #13x13 -> 13x13
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), #13x13 -> 13x13
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), #13x13 -> 13x13
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2), # #13x13 -> 6x6
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((6,6))

        self.classifier = nn.Sequential(
            nn.Dropout(), #prevent overfitting : 關閉特定神經元，讓剩下的學會提取特徵
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return(x)

    #     #padding 計算範例：
    #     #55 = floor[(224 - 11+2p)/4 ]+1 => 54 = floor[ (213+2p)/4 ] => 213+2p > 216 => p>1.5 取整數 p=2
    #     self.conv1 = nn.Conv2d(in_channels, 96,kernel_size=11, stride=4, padding=2) #224x224 -> 55x55
    #     #nn.ReLU(inplace=True)
    #     self.conv2 = nn.Conv2d(96, 256,kernel_size=5,  padding=2) #(96)x55x55 -> (256)x27x27
    #     self.conv3 = nn.Conv2d(256, 384,kernel_size=3,  padding=1) #27x27 -> 13x13
    #     self.conv4 = nn.Conv2d(384, 384,kernel_size=3,  padding=1) #13x13 ->13x13
    #     self.conv5 = nn.Conv2d(384, 256,kernel_size=3,  padding=1) #13x13 -> 13x13
    #     #13x13 -> 2x2048
    #     #2x2048 -> 2x2048
    #     #2x2048 -> 1000
    # def forward(self, x):


In [ ]:
model = AlexNet(in_channels=3, num_classes=10).to('cuda')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

In [5]:
model.train()
for epoch in range(30):
    for batch_idx, (data, targets) in enumerate(tqdm(train_loader)):
        data = data.to('cuda')
        targets = targets.to('cuda')

        scores = model(data)

        loss = criterion(scores, targets)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

100%|██████████| 391/391 [00:35<00:00, 11.00it/s]


In [6]:
num_correct = 0
num_samples = 0
model.eval()

with torch.no_grad():
    for x,y in test_loader:
        x = x.to('cuda')
        y = y.to('cuda')

        scores = model(x)

        _, predictions = scores.max(1)
        num_correct += (predictions == y).sum() #配對索引
        num_samples += predictions.size(0)

print(num_correct / num_samples)

tensor(0.8226, device='cuda:0')
